# DeepForge — PyTorch training session and checkpoint recovery

A real CPU-friendly classification example using a small PyTorch MLP and distinct deterministic synthetic training and validation tensors. Demonstrates inline dashboard, optional local native desktop window, validation, periodic checkpoints, safe stop, and explicit restore. No dataset downloads are required. Metrics are computed by the model, not simulated. This is not a benchmark.

## 1. Install

Run if needed, then restart the kernel after upgrading:

```python
%pip install "deepforge-research-toolkit[interactive,resources]==0.7.1.1.3"
```

Install a compatible PyTorch build using https://pytorch.org/get-started/locally/ if the kernel does not already provide torch. In Colab, optionally enable custom widget management with `from google.colab import output; output.enable_custom_widget_manager()`. Do not replace a working hosted CUDA build. When the exact package version is unavailable, install the matching repository source as described in README.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4
import torch
from deepforge.notebook_setup import build_training_session

torch.manual_seed(42)
device = torch.device("cpu")  # Set to cuda only in a compatible environment.
EPOCHS, BATCH_SIZE = 5, 32
OPEN_DESKTOP = False  # True only for local VS Code/Jupyter with Tkinter + graphical display.
SHOW_WIDGETS = False  # True if ipywidgets frontend support is available.
DEMO_STOP = True  # Pause safely after epoch 2, then restore in a separate cell.

generator = torch.Generator().manual_seed(42)
X = torch.randn(256, 4, generator=generator)
Y = (X[:, 0]*X[:, 1] + 0.5*X[:, 2] > 0).long()
X_val = torch.randn(128, 4, generator=generator)
Y_val = (X_val[:, 0]*X_val[:, 1] + 0.5*X_val[:, 2] > 0).long()
batches = len(X)//BATCH_SIZE
X, Y, X_val, Y_val = [a.to(device) for a in (X, Y, X_val, Y_val)]
criterion = torch.nn.CrossEntropyLoss()

def make_model():
    model = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.ReLU(),
                                torch.nn.Linear(16, 2)).to(device)
    return model, torch.optim.Adam(model.parameters(), lr=0.02)


In [ ]:
# A reproducible setup without requiring a widget click.
# Alternatively use deepforge.notebook_setup.setup_panel(config_path)
# and save a compatible configuration before building the session.
run_id = "torch_" + uuid4().hex[:10]
storage = Path("deepforge_demo_runs").resolve()
storage.mkdir(exist_ok=True)
config_path = storage / (run_id + ".json")
config = dict(storage=str(storage), keep_last=2, periodic=True, theme="jade",
              max_epochs=20, interval=0.5, audio=False, show_diagnostics=True,
              show_resources=True, alert_patience=3, widget_scroll=False,
              gradient_threshold=100.0,
              epoch_metrics=["loss","val_loss","accuracy","val_accuracy","lr","gradient_norm"],
              overall_fields=["Status","Epoch","Batch","Elapsed (s)","ETA (s)","Batches/s"])
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

def build():
    model, optimizer = make_model()
    monitor, view, checkpoint, session = build_training_session(
        config_path, run_id=run_id, epochs=EPOCHS, batches_per_epoch=batches,
        model=model, optimizer=optimizer, experiment_metadata={
            "dataset":"synthetic binary classification", "seed":42,
            "train_samples":len(X), "validation_samples":len(X_val),
            "model":"MLP16", "batch_size":BATCH_SIZE})
    return model, optimizer, monitor, view, checkpoint, session

model, optimizer, monitor, view, checkpoint, session = build()
if SHOW_WIDGETS:
    session.controls()
if OPEN_DESKTOP:
    desktop = session.open_desktop()
view.refresh(force=True)
print("Run ID:", run_id, "Snapshot:", monitor.path.resolve())


In [ ]:
def train(start_epoch=1, global_step=0, pause_demo=False):
    assert monitor.completed == (start_epoch - 1)*batches
    try:
        for epoch in range(start_epoch, EPOCHS + 1):
            model.train()
            loss_sum = correct = seen = 0
            for batch, offset in enumerate(range(0, len(X), BATCH_SIZE), 1):
                xx, yy = X[offset:offset+BATCH_SIZE], Y[offset:offset+BATCH_SIZE]
                optimizer.zero_grad(set_to_none=True)
                logits = model(xx)
                loss = criterion(logits, yy)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 100).item()
                optimizer.step()
                global_step += 1
                seen += len(xx)
                loss_sum += loss.item()*len(xx)
                correct += (logits.argmax(1) == yy).sum().item()
                monitor.batch(epoch=epoch, batch=batch, samples=len(xx), metrics={
                    "loss":loss_sum/seen, "accuracy":100*correct/seen,
                    "lr":optimizer.param_groups[0]["lr"], "gradient_norm":grad_norm})
                session.poll_control()
                view.refresh()
            model.eval()
            monitor.record_stage(epoch, "validation", "running", completed=0, total=1)
            with torch.no_grad():
                logits = model(X_val)
                val_loss = criterion(logits, Y_val).item()
                val_acc = 100*(logits.argmax(1) == Y_val).float().mean().item()
            monitor.epoch_end(epoch, {"loss":val_loss,"accuracy":val_acc})
            monitor.record_stage(epoch, "validation", "completed", completed=1, total=1)
            if pause_demo and epoch == 1:
                session.request_stop(after_next=True)
            if session.epoch_boundary(epoch, global_step=global_step):
                return global_step
            checkpoint.save_epoch(epoch, global_step=global_step, metadata=session.metadata)
            view.refresh(force=True)
        monitor.finish()
        return global_step
    except Exception:
        if monitor.status == "running":
            monitor.finish(failed=True)
        raise
    finally:
        view.refresh(force=True)

step = train(pause_demo=DEMO_STOP)
print("Status:", monitor.status, "Completed batches:", monitor.completed)


## 4. Resume the paused run

This cell uses fresh model/optimizer/session objects and the exact same run ID, configuration, data and architecture. The notebook's deterministic demonstration pauses after epoch 2. In a new runtime you must recreate the original data and settings and retain the original run ID and config; do not generate a different ID. For cloud recovery store them in durable storage. Only restore a trusted checkpoint.

In [ ]:
if monitor.status == "paused":
    model, optimizer, monitor, view, checkpoint, session = build()
    state = session.restore()
    print("Restored epoch:", state["epoch"])
    if SHOW_WIDGETS:
        session.controls()
    step = train(start_epoch=state["epoch"]+1, global_step=state["global_step"])
else:
    print("No paused session; skip restore.")
print("Final status:", monitor.status)
if monitor.status == "completed":
    assert monitor.completed == EPOCHS*batches
    assert len(monitor.epoch_history) == EPOCHS


## 5. Which interface is where?

- Inline `NotebookMonitor`: notebook cell output, no graphical desktop needed.
- `session.open_desktop()`: local independent native window with controls. Keep OPEN_DESKTOP=False in hosted Colab/Kaggle. The existing window can continue reading the same snapshot after explicit restore.
- `session.controls()`: optional notebook widgets; delivery of button callbacks during a blocking cell is frontend dependent.
- `deepforge-studio`: separate local research/data auditing desktop application.
- A manual or safe-stop checkpoint is verified at a completed epoch boundary; exact reproducibility across hardware or nondeterministic operations is not guaranteed. The programmatic stop above is intentionally independent of browser widget callbacks.